# 02 — Relancer les premières baselines classiques

Contribution partielle à **L04, L07 et L12**. Objectif : relier préparation, entraînement, validation et vérification des sauvegardes. Une **baseline** est une référence de comparaison ; ici ce sont des classifieurs classiques, pas des LLM.

Prérequis : environnement verrouillé installé selon `docs/20_environnement_python.md`. Le contrôle initial utilise uniquement des textes inventés. Les runs réels sont désactivés par défaut et exigent les sources privées locales. Aucun poids neuronal, dépendance ou corpus n'est téléchargé par ce notebook. Le noyau peut avoir un autre Python : nous appelons les scripts par uv, hors ligne.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/project_uv.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir le notebook depuis le dépôt ou son dossier notebooks/.')
sys.path.insert(0, str(ROOT / 'scripts'))
from notebook_support import CONFIGS, run_project, train_baselines

# off : pas d'entraînement réel ; smoke : petit test ; full : smoke vérifié puis run complet.
TRAINING_MODE = os.environ.get('TUNISCOPE_BASELINE_MODE', 'off')
FAMILY = os.environ.get('TUNISCOPE_BASELINE_FAMILY', 'characters')
if TRAINING_MODE not in {'off', 'smoke', 'full'} or FAMILY not in CONFIGS:
    raise ValueError('Choisir off/smoke/full et characters/words/combined')
check = json.loads(run_project('check_environment.py', capture=True))
assert check['corpus_read'] is False
print(json.dumps(check, indent=2))
print('Entraînement réel :', TRAINING_MODE, '| représentation :', FAMILY)

## Les étapes de l'entraînement

1. Charger les mêmes partitions **v2**. Dans le mode smoke, sélectionner 100 lignes par partition/tâche ; ce petit test n'est pas un résultat scientifique final.
2. Construire TF-IDF sur **train seulement** : transformer les textes en colonnes numériques pondérées par fréquence. `characters` utilise des morceaux de 3 à 5 caractères ; `words` des suites de 1 à 2 mots ; `combined` concatène les deux puis applique une normalisation globale L2. Ce sont trois configurations figées, pas une recherche de paramètres.
3. Ajuster une **régression logistique**, classifieur qui apprend les poids reliant ces colonnes aux labels. Un modèle distinct est entraîné par tâche et variante brut/normalisé. La référence majoritaire prédit toujours la classe la plus fréquente du train.
4. Prédire les labels de validation sans y réapprendre TF-IDF ou le classifieur. Calculer notamment **macro-F1**, moyenne du F1 des classes : un score de qualité, pas un pourcentage d'avancement.
5. Sauvegarder modèles, prédictions et métadonnées dans un **nouveau dossier ignoré par Git**. Recharger les modèles et recalculer les métriques avec `verify_baseline_run.py`. Un échec de vérification arrête la suite ; les anciens runs restent intacts.

Pour lancer réellement : remplacer TRAINING_MODE ci-dessus par `'smoke'`, puis éventuellement `'full'`. Le mode full relance d'abord un smoke et ne poursuit qu'après vérification. Une expérience complète produit six résultats : deux tâches × (majorité brute + TF-IDF brut + TF-IDF normalisé). Le code d'entraînement reste dans `scripts/run_baselines.py` et `scripts/baseline_models.py`, pas recopié dans le notebook.

In [ ]:
if TRAINING_MODE == 'off':
    runs = []
    print('Aucun corpus lu : seul le contrôle fictif a été exécuté.')
else:
    runs = train_baselines(FAMILY, TRAINING_MODE)
    print(json.dumps(runs, ensure_ascii=False, indent=2))
    print('Résultats vérifiés. Consigner ce run significatif dans EXPERIMENTS.md ; ne pas publier les modèles/predictions privés.')

## Comment interpréter la suite ?

Un contrôle fictif réussi ne mesure pas TSAC ou MADAR. Un smoke réussi montre que le pipeline fonctionne sur un sous-ensemble ; ses scores ne remplacent pas ceux du run complet. Un run complet utilise toujours la **validation**, pas le test final. Aucun choix de label ou de nettoyage n'est modifié ici en fonction du score.

Les résultats historiques complets sont documentés dans les rapports et docs 23–25 : caractères seuls restent la meilleure référence initiale. Les règles seules (doc 29) puis l'ajout lexical aux caractères (doc 30) n'ont pas dépassé cette référence ; ces résultats négatifs sont conservés. Ce notebook n'automatise pas encore ces deux familles : leurs scripts et vérificateurs dédiés restent la référence.

**L04 n'est pas terminé** : BiGRU et modèle de langage caractères demandés restent à réaliser dans une phase autorisée distincte. Trois graines finales, bootstrap, test officiel et validation cloud restent également ouverts. Ne pas charger de checkpoint joblib reçu d'une source non fiable.

Avant de partager le notebook : effacer ses sorties. Les fichiers livrés ici sont sans sorties ; les nouveaux modèles/prédictions restent sous `outputs/baselines/notebooks/`. Procédure et périmètre vérifié : `docs/31_notebooks_reproductibles.md`.